# 03 · HAC inference, or: is that mean actually distinguishable from zero?

A mean RankIC of 0.03 can earn a `pass` whether it came from 600 periods or six.
That is not a threshold problem, it is a missing standard error.

Two things make it worse in this specific setting:

1. **Overlap the design creates.** With h-period forward returns sampled every
   period, consecutive observations share h−1 periods *by construction*. They
   are not independent draws, and treating them as such shrinks the standard
   error you report.
2. **Autocorrelation in the signal itself.**

This notebook covers the correction, why its bandwidth must be pre-registered,
and how to turn a result into a number with currency attached.

In [ ]:
# Put `backend/` on sys.path so `app.*` imports work regardless of where
# Jupyter was launched from. Walks up until it finds the backend package.
import sys, pathlib

here = pathlib.Path.cwd()
for candidate in (here, *here.parents):
    if (candidate / "backend" / "app").is_dir():
        sys.path.insert(0, str(candidate / "backend"))
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("could not locate backend/ — run from inside the repo")

FIXTURES = REPO_ROOT / "backend" / "tests" / "fixtures" / "edgar"
print("repo root :", REPO_ROOT)
print("fixtures  :", FIXTURES, "(exists)" if FIXTURES.is_dir() else "(MISSING)")

## Naive vs HAC on autocorrelated data

If the HAC t-statistic is *not* materially smaller than the naive one on an
autocorrelated series, the correction is not being applied and every verdict
built on it is overstated.

In [ ]:
import numpy as np
import pandas as pd

from app.factor_validation.inference import naive_iid_tstat, newey_west_mean_tstat

def ar1(n, phi, mean=0.02, sigma=0.05, seed=7):
    """AR(1) series with a planted mean — autocorrelated by construction."""
    rng = np.random.default_rng(seed)
    shocks = rng.normal(0.0, sigma, n)
    out = np.empty(n)
    out[0] = shocks[0]
    for i in range(1, n):
        out[i] = phi * out[i - 1] + shocks[i]
    return pd.Series(out + mean)

print(f"{'phi':>6} {'naive t':>10} {'HAC t':>10} {'HAC/naive':>11}  interpretation")
for phi in (0.0, 0.3, 0.7):
    s = ar1(400, phi)
    naive = naive_iid_tstat(s)
    hac = newey_west_mean_tstat(s, holding_periods=1)["tstat"]
    print(f"{phi:>6} {naive:>10.2f} {hac:>10.2f} {hac/naive:>11.2f}  "
          f"{'independent' if phi == 0 else 'persistent — naive overstates'}")

At `phi=0` the two barely differ; as persistence rises the naive statistic
inflates. The correction is doing work, and the direction is always the same —
naive is optimistic, never pessimistic.

## The lag rule, and why it must be pre-registered

Bandwidth is `max(floor(4·(n/100)^(2/9)), holding_periods − 1)`. The automatic
part is the Newey–West 1994 rule; the floor handles the overlap the sampling
design creates.

**`holding_periods` is a pre-registration item.** Tuning the lag until the
t-statistic clears 2 is p-hacking with extra steps.

In [ ]:
from app.factor_validation.inference import lag_sensitivity, newey_west_lag

print("automatic bandwidth by sample size (holding_periods=1):")
for n in (50, 100, 250, 400, 1000):
    print(f"  n={n:>5}  lag={newey_west_lag(n)}")

print("\nthe holding-period floor (n=250):")
for h in (1, 5, 21):
    print(f"  holding_periods={h:>3}  lag={newey_west_lag(250, h)}")

print("\nsensitivity to bandwidth — a diagnostic, NOT a menu to choose from:")
s = ar1(400, 0.7)
for row in lag_sensitivity(s, [0, 2, 4, 6, 12], holding_periods=1):
    print(f"  lags={row['lags']:>3}  t={row['tstat']:>7.3f}")

That last table is exactly what a reader should be shown *and* exactly what an
author must not shop through. Publishing it alongside a pre-registered choice is
the difference between transparency and selection.

## Incremental signal value

The central statistic of this track: not "is the enriched arm good?" but "does
it add anything over the baseline?" Both arms span the same periods and carry
the same trading assumptions, so cost largely cancels in the contrast.

A positive mean with |t| < 2 is not a finding. It is a shrug.

In [ ]:
from app.factor_validation.inference import incremental_signal_value

rng = np.random.default_rng(11)
n = 300
baseline = pd.Series(rng.normal(0.010, 0.05, n))      # price-only arm

for label, lift in (("no real lift", 0.000), ("genuine lift", 0.012)):
    enriched = baseline + rng.normal(lift, 0.02, n)   # price + text arm
    result = incremental_signal_value(enriched, baseline, holding_periods=1)
    verdict = (
        "shrug" if result["tstat"] is None or abs(result["tstat"]) < 2
        else "distinguishable from zero"
    )
    print(f"{label:14} mean={result['mean']:+.5f}  t={result['tstat']:+.2f}  -> {verdict}")

## Attaching currency: net economic value and break-even capital

Units must match before subtracting. Returns arrive in basis points; inference
cost arrives in currency:

```
net economic value (bps)
  = gross excess return (bps)
  − trading cost (bps)
  − inference cost / capital × 10,000
```

The consequence is the interesting part: inference cost is **fixed** while
capital is the **denominator**, so the same model is uneconomic at small size
and free at large size. "Which tier wins" is the wrong question; "above what AUM
does this tier pay for itself" is the right one.

In [ ]:
from app.factor_validation.inference import breakeven_capital, net_economic_value_bps

GROSS_BPS, TRADING_BPS, INFERENCE_COST = 45.0, 12.0, 2_000.0

print(f"gross {GROSS_BPS} bps, trading cost {TRADING_BPS} bps, "
      f"inference ${INFERENCE_COST:,.0f}/month\n")
print(f"{'AUM':>16} {'inference drag':>16} {'net':>10}")
for aum in (1e6, 5e6, 25e6, 100e6, 500e6):
    r = net_economic_value_bps(GROSS_BPS, TRADING_BPS, INFERENCE_COST, aum)
    print(f"{aum:>16,.0f} {r['inference_cost_bps']:>15.2f}  {r['net_economic_value_bps']:>9.2f}")

be = breakeven_capital(GROSS_BPS, TRADING_BPS, INFERENCE_COST)
print(f"\nbreak-even capital: ${be:,.0f}")
print("below this the extraction costs more than the edge it finds")

## What this machinery refuses to do

Unavailable results carry an `unavailable_reason` and **never** substitute zero —
the same contract the rest of the platform uses. A missing number stays missing
and says why.

In [ ]:
short = pd.Series([0.01, 0.02, -0.01])          # below the minimum
flat = pd.Series([0.01] * 50)                    # no variance

for label, series in (("3 observations", short), ("zero variance", flat)):
    r = newey_west_mean_tstat(series)
    print(f"{label:16} tstat={r['tstat']}  reason={r['unavailable_reason']!r}")